In [ ]:
import os 
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
os.environ['XLA_FLAGS'] = '--xla_gpu_cuda_data_dir=/usr/lib/cuda'
import numpy as np
import pandas as pd
import pdb
import tensorflow as tf
import sys
# from IPython.display import clear_output
import time
from src.gan.lib import models
from src.gan.lib import utils
from utilities.UTR5_utilities import UTRData, Condition2Tensor, Seq2Tensor, make_res, seqprep,  CODES_2, CELLTYPE_CODES_UTR5
from utilities.pl_regressor import RNARegressor
from utilities.legnet_softclass import LegNetClassifier
import socket
import datetime
from tqdm import tqdm
import random
import matplotlib.pyplot as plt
import argparse


import torch
from torch.utils.data import DataLoader, Dataset

import tensorflow.keras.backend as K
from tensorflow.keras.optimizers import Adam

In [ ]:
print("Using:", tf.test.gpu_device_name())

In [3]:
tf.compat.v1.enable_eager_execution()

In [4]:
MODEL_NAME = 'WGAN-TF2-UTR5-16-oversampled'
OUTPUT_PATH = os.path.join('outputs', MODEL_NAME)
TRAIN_LOGDIR = os.path.join("logs_", "tensorflow", MODEL_NAME, 'train_data') # Sets up a log directory.
if not os.path.exists(OUTPUT_PATH):
    os.makedirs(OUTPUT_PATH)

In [5]:
def plot(x, y, logdir, name, xlabel=None, ylabel=None, title=None):

    plt.plot(x,y,'-')
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.savefig(logdir+'/'+name+'.png')
    plt.clf()


def plot_valid(x1, y1, x2, y2, logdir, name, xlabel=None, ylabel=None, title=None):

    plt.plot(x2,y2,'-',color='tab:blue')
    plt.plot(x1,y1,'-',color='tab:orange')

    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)

    plt.savefig(logdir+'/'+name+'.png')
    plt.clf()    

def gradient_penalty_loss( y_true, y_pred, discriminator):
  """
  Computes gradient penalty based on prediction and weighted real / fake samples
  """
  alpha = K.random_uniform((DIM, 1, 1))
  averaged_samples = (alpha * y_pred) + ((1 - alpha) * y_true)

  gradients = K.gradients(y_pred, averaged_samples)[0]
  # compute the euclidean norm by squaring 
  gradients_sqr = K.square(gradients)
  gradients_sqr_sum = K.sum(gradients_sqr,
                            axis=np.arange(1, len(gradients_sqr.shape)))
  gradient_l2_norm = K.sqrt(gradients_sqr_sum)
  # compute lambda * (1 - ||grad||)^2 still for each single sample
  gradient_penalty = K.square(1 - gradient_l2_norm)
  return K.mean(gradient_penalty)

def log(samples_dir=False,suff=None):
    stamp = datetime.date.strftime(datetime.datetime.now(), "%Y.%m.%d-%Hh%Mm%Ss") + "_{}".format(socket.gethostname())
    full_logdir = os.path.join("./logs/", stamp)
    if suff:
        full_logdir = full_logdir + suff
    os.makedirs(full_logdir, exist_ok=True)
    if samples_dir: os.makedirs(os.path.join(full_logdir, "samples"), exist_ok=True)
    log_dir = "{}:{}".format(socket.gethostname(), full_logdir)
    
    return full_logdir, 0

In [6]:
logdir, checkpoint_baseline = log(samples_dir=True)
logdir2 = ''

In [7]:
UTR_LEN = 50
rna_vocab = {"A":0,
             "C":1,
             "G":2,
             "T":3,
             "*":4}

rev_rna_vocab = {v:k for k,v in rna_vocab.items()}

In [8]:
BATCH_SIZE = 128 # Batch size
ITERS = 3000 # How many iterations to train for
SEQ_LEN = UTR_LEN # Sequence length in characters
DIM = 16 # Model dimensionality.
CRITIC_ITERS = 5 # How many critic iterations per generator iteration. 
LAMBDA = 10 # Gradient penalty lambda hyperparameter.
LR = 1e-4
LAMBDA = 10 # For gradient penalty

CURRENT_EPOCH = 1 # Epoch start from
SAVE_EVERY_N_EPOCH = 500 # Save checkpoint at every n epoch
MIN_LR = 0.000001 # Minimum value of learning rate
DECAY_FACTOR=1.00004 # learning rate decay factor
'''
Set seed for reproducibility
'''
seed = 42
np.random.seed(seed)
tf.random.set_seed(seed)

In [9]:
'''
Build GAN
'''
model_type = "resnet"
data_enc_dim = 5
data_size = SEQ_LEN * data_enc_dim
# data_size = 256
gen_layers = 3
disc_layers = 3
lmbda = 10. #lipschitz penalty hyperparameter.

SAMPLE_SIZE = 128

N_CHANNELS = DIM

In [10]:
file_writer = tf.summary.create_file_writer(TRAIN_LOGDIR)


In [11]:
G = models.resnet_g2(DIM,N_CHANNELS,SEQ_LEN,5,res_layers=gen_layers)
D = models.resnet_d2(N_CHANNELS,SEQ_LEN,5,res_layers=disc_layers)

G.summary()
D.summary()

D_optimizer = Adam(learning_rate=LR, beta_1=0.5, beta_2=0.99)
G_optimizer = Adam(learning_rate=LR, beta_1=0.5, beta_2=0.99)

EPOCHs = ITERS

Model: "Generator"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 16)]         0           []                               
                                                                                                  
 dense (Dense)                  (None, 800)          13600       ['input_1[0][0]']                
                                                                                                  
 reshape (Reshape)              (None, 50, 16)       0           ['dense[0][0]']                  
                                                                                                  
 activation (Activation)        (None, 50, 16)       0           ['reshape[0][0]']                
                                                                                          

In [12]:
@tf.function
def WGAN_GP_train_d_step(real_sample, batch_size, step):

    noise = tf.random.normal([batch_size, DIM])
    epsilon = tf.random.uniform(shape=[batch_size, 1, 1], minval=0, maxval=1)
    ###################################
    # Train D
    ###################################
    with tf.GradientTape(persistent=True) as d_tape:
        with tf.GradientTape() as gp_tape:
            fake_sample = G([noise], training=True)
            fake_sample_mixed = epsilon * tf.dtypes.cast(real_sample, tf.float32) + ((1 - epsilon) * fake_sample)
            fake_mixed_pred = D([fake_sample_mixed], training=True)
            
        # Compute gradient penalty
        grads = gp_tape.gradient(fake_mixed_pred, fake_sample_mixed)
        grad_norms = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=[1,2])) # Originally axis=[1,2]
        # grad_norms = tf.norm(grads, axis=[1,2])
        gradient_penalty = tf.reduce_mean(tf.square(grad_norms - 1.))
        
        fake_pred = D([fake_sample], training=True)
        real_pred = D([real_sample], training=True)
        
        D_loss = tf.reduce_mean(fake_pred) - tf.reduce_mean(real_pred) + LAMBDA * gradient_penalty
    # Calculate the gradients for discriminator
    D_gradients = d_tape.gradient(D_loss,D.trainable_variables)
    # Apply the gradients to the optimizer
    D_optimizer.apply_gradients(zip(D_gradients,D.trainable_variables))
    # Write loss values to tensorboard
    if step % 10 == 0:
        with file_writer.as_default():
            tf.summary.scalar('D_loss', tf.reduce_mean(D_loss), step=step)

    return D_loss, gradient_penalty

@tf.function
def WGAN_GP_train_g_step(real_sample, batch_size, step):

    noise = tf.random.normal([batch_size, DIM])
    ###################################
    # Train G
    ###################################
    with tf.GradientTape() as g_tape:
        fake_sample = G([noise], training=True)
        fake_pred = D([fake_sample], training=True)
        G_loss = -tf.reduce_mean(fake_pred)

    G_gradients = g_tape.gradient(G_loss,
                                            G.trainable_variables)
    # Apply the gradients to the optimizer
    G_optimizer.apply_gradients(zip(G_gradients,
                                                G.trainable_variables))
    # Write loss values to tensorboard
    if step % 10 == 0:
        with file_writer.as_default():
            tf.summary.scalar('G_loss', G_loss, step=step)

    return G_loss, noise

checkpoint_path = os.path.join("checkpoints", "tensorflow", MODEL_NAME)

ckpt = tf.train.Checkpoint(generator=G,
                        discriminator=D,
                        G_optimizer=G_optimizer,
                        D_optimizer=D_optimizer)

ckpt_manager = tf.train.CheckpointManager(ckpt, checkpoint_path, max_to_keep=40)


def generate_and_save_samples(model, epoch, test_input, figure_size=(12,6), subplot=(3,6), save=True, is_flatten=False):
    '''
        Generate samples and plot it.
    '''
    predictions = model.predict(test_input)
    utils.save_samples(logdir, predictions, epoch, rev_rna_vocab, annotated=False)


def generate_and_save_samples_and_noise(model, total_samples, epoch, 
                                        batch_size=50000, save=True, is_flatten=False):
    """
    Generate samples from noise in batches and save both sequences and noise vectors.
    
    """
    latent_vectors = []
    sequences = []

    num_batches = (total_samples + batch_size - 1) // batch_size

    for i in tqdm(range(num_batches), desc=f"Generating {total_samples} samples"):
        current_batch_size = min(batch_size, total_samples - i * batch_size)
        sample_noise = tf.random.normal([current_batch_size, DIM])
        predictions = model(sample_noise, training=False).numpy()

        argmax = np.argmax(predictions, axis=2)
        batch_sequences = ["".join(rev_rna_vocab[d] for d in line) for line in argmax]

        latent_vectors.extend(sample_noise.numpy())
        sequences.extend(batch_sequences)

    if save:
        df = pd.DataFrame({
            "latent_vector": list(latent_vectors),
            "sequence": sequences
        })
        os.makedirs(os.path.join(logdir, "samples"), exist_ok=True)
        out_file = os.path.join(logdir, "samples", f"samples_and_noise_{epoch}.pkl")
        df.to_pickle(out_file)
        print(f"[+] Saved noise + samples → {out_file}")

In [13]:
G = models.resnet_g2(DIM, N_CHANNELS, SEQ_LEN, 5, res_layers=gen_layers)
D = models.resnet_d2(N_CHANNELS, SEQ_LEN, 5, res_layers=disc_layers)

G_optimizer = Adam(learning_rate=LR, beta_1=0.5, beta_2=0.99)
D_optimizer = Adam(learning_rate=LR, beta_1=0.5, beta_2=0.99)


checkpoint = tf.train.Checkpoint(generator=G,
                                 discriminator=D,
                                 g_optimizer=G_optimizer,
                                 d_optimizer=D_optimizer)


checkpoint_dir = "checkpoints/tensorflow/WGAN-TF2-UTR5-16-oversampled"

latest_ckpt = tf.train.latest_checkpoint(checkpoint_dir)
print("Latest checkpoint found:", latest_ckpt)

status = checkpoint.restore(latest_ckpt)

status.expect_partial() 
print(f"Model restored from{latest_ckpt}")

G.summary()
D.summary()


Latest checkpoint found: checkpoints/tensorflow/WGAN-TF2-UTR5-16-oversampled/ckpt-8
Model restored fromcheckpoints/tensorflow/WGAN-TF2-UTR5-16-oversampled/ckpt-8
Model: "Generator"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_3 (InputLayer)           [(None, 16)]         0           []                               
                                                                                                  
 dense_2 (Dense)                (None, 800)          13600       ['input_3[0][0]']                
                                                                                                  
 reshape_1 (Reshape)            (None, 50, 16)       0           ['dense_2[0][0]']                
                                                                                                  
 activation_12 (Activation)

In [14]:
TOTAL_SAMPLES = 10_000
CHUNK_SIZE = 10_000
BATCH_SIZE = 2_500

In [15]:
for chunk_idx in range(TOTAL_SAMPLES // CHUNK_SIZE):
    print(f"--- Generating chunk {chunk_idx}/{TOTAL_SAMPLES // CHUNK_SIZE} ---")
    generate_and_save_samples_and_noise(
        model=G,
        total_samples=CHUNK_SIZE,
        epoch=chunk_idx,
        batch_size=BATCH_SIZE,
        save=True
    )

--- Generating chunk 0/1 ---


Generating 10000 samples: 100%|██████████| 4/4 [00:00<00:00, 10.97it/s]


[+] Saved noise + samples → ./logs/2026.05.02-21h55m12s_Sydney/samples/samples_and_noise_0.pkl


In [ ]:


out_file = "data/gan_generated_sequences_UTR5_10k.csv"
first = True

for chunk_idx in tqdm(range(TOTAL_SAMPLES // CHUNK_SIZE), desc="concating chunks"):
    file_path = f"logs/your log path/samples/samples_and_noise_{chunk_idx}.pkl"
    df_chunk = pd.read_pickle(file_path)
    df_chunk.to_csv(out_file, mode="a", header=first, index=False)
    first = False  

print("[+] Data saved to ", out_file)
